# Corporate Reorganization — Retriever evaluation

This notebook supports two execution modes:

1. **Local full ablation** using your local `legalpacaenv`.
   - This can run `bm25_flat` because it uses your local Pyserini and local Java.
2. **SageMaker Processing** using `processing_eval/run_eval_sm.py`.
   - The default `HuggingFaceProcessor` path is appropriate for the dense systems.
   - `bm25_flat` on SageMaker Processing needs a custom image or extended container that includes Java + Pyserini.

Before evaluation:
- rebuild `../data/final_annotations_gold/processed/`
- upload that rebuilt processed directory to S3
- set the structured and flat model artifact URIs below


In [1]:
import os
import sys
import time
from pathlib import Path

import sagemaker
from dotenv import find_dotenv, load_dotenv
from sagemaker.huggingface import HuggingFaceProcessor
from sagemaker.processing import ProcessingInput, ProcessingOutput

modernbert_dir = Path("../modernbert").resolve()
if str(modernbert_dir) not in sys.path:
    sys.path.insert(0, str(modernbert_dir))

from processing_eval.experiment import build_default_system_specs, run_retrieval_experiment

load_dotenv(find_dotenv(usecwd=True))

role = os.environ["SAGEMAKER_EXECUTION_ROLE_ARN"]
session = sagemaker.Session()
bucket = session.default_bucket()
prefix = "corporate_reorganization/retriever"

print("role:", role)
print("bucket:", bucket)
print("prefix:", prefix)
print("modernbert_dir:", modernbert_dir)


sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/xdg-ubuntu/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/lbrenap/.config/sagemaker/config.yaml
role: arn:aws:iam::371087393859:role/defaultrole
bucket: sagemaker-us-east-1-371087393859
prefix: corporate_reorganization/retriever
modernbert_dir: /home/lbrenap/Documents/projects/legalpaca/corporate_reorganization/modernbert


In [2]:
# Reuse your existing structured model here if you do not retrain it.
structured_model_s3_uri = "s3://sagemaker-us-east-1-371087393859/huggingface-pytorch-training-2026-03-16-00-09-29-909/output/model.tar.gz"
#s3://sagemaker-us-east-1-371087393859/huggingface-pytorch-training-2026-01-05-14-15-44-813/output/model.tar.gz")

# Replace this after training the flat model.
flat_model_s3_uri = "s3://sagemaker-us-east-1-371087393859/huggingface-pytorch-training-2026-03-15-23-12-42-656/output/model.tar.gz"

split = "test"
regimes = ["same_case_legacy", "same_case_full", "global_split"]

# Local full ablation includes BM25.
local_output_dir = Path("../test_results/retrieval_ablation_local").resolve()

# SageMaker Processing path below is dense-only unless you build a custom image with Java + Pyserini.
processing_systems = "dense_open_flat,base_modernbert_flat,fine_tuned_flat,fine_tuned_structured"
processing_timestamp = time.strftime("%Y%m%d_%H%M%S", time.gmtime())
processing_output_s3_uri = f"s3://{bucket}/{prefix}/eval_processing/{processing_timestamp}"

print("structured_model_s3_uri:", structured_model_s3_uri)
print("flat_model_s3_uri:", flat_model_s3_uri)
print("local_output_dir:", local_output_dir)
print("processing_output_s3_uri:", processing_output_s3_uri)


structured_model_s3_uri: s3://sagemaker-us-east-1-371087393859/huggingface-pytorch-training-2026-03-16-00-09-29-909/output/model.tar.gz
flat_model_s3_uri: s3://sagemaker-us-east-1-371087393859/huggingface-pytorch-training-2026-03-15-23-12-42-656/output/model.tar.gz
local_output_dir: /home/lbrenap/Documents/projects/legalpaca/corporate_reorganization/test_results/retrieval_ablation_local
processing_output_s3_uri: s3://sagemaker-us-east-1-371087393859/corporate_reorganization/retriever/eval_processing/20260316_021420


In [3]:
processed_dir = Path("../data/final_annotations_gold/processed").resolve()
assert processed_dir.exists(), f"Missing processed_dir: {processed_dir}"

# Keep this prefix unchanged to overwrite the existing processed dataset in S3.
# Change it if you want a side-by-side versioned copy instead.
data_key_prefix = f"{prefix}/data/processed_ablation_v2"

data_s3_uri = session.upload_data(
    path=str(processed_dir),
    bucket=bucket,
    key_prefix=data_key_prefix,
)
inputs = {"data": data_s3_uri}

print("processed_dir:", processed_dir)
print("data_s3_uri:", data_s3_uri)
print("Re-run this cell after rebuilding processed/ to overwrite the same S3 keys.")


processed_dir: /home/lbrenap/Documents/projects/legalpaca/corporate_reorganization/data/final_annotations_gold/processed
data_s3_uri: s3://sagemaker-us-east-1-371087393859/corporate_reorganization/retriever/data/processed_ablation_v2
Re-run this cell after rebuilding processed/ to overwrite the same S3 keys.


In [4]:
# Local full ablation. This is the easiest path for BM25 because it uses your local Java + Pyserini.
systems = build_default_system_specs(
    structured_model_s3_uri=structured_model_s3_uri,
    flat_model_s3_uri=flat_model_s3_uri,
    include_cohere_flat=False,
)

local_results = run_retrieval_experiment(
    processed_dir=processed_dir,
    output_dir=local_output_dir,
    split=split,
    systems=systems,
    regimes=regimes,
    max_len_query=4096,
    max_len_passage=600,
    query_batch_size=64,
    passage_batch_size=256,
    ks=(1, 5, 10, 20),
)

local_output_dir


Mar 15, 2026 10:14:25 PM org.apache.lucene.store.MemorySegmentIndexInputProvider <init>
INFO: Using MemorySegmentIndexInput with Java 21; to disable start with -Dorg.apache.lucene.store.MMapDirectory.enableMemorySegments=false

2026-03-15 22:14:26,460 - huggingface_hub.utils._http - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


config.json:   0%|          | 0.00/650 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/314 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
decoder.bias      | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


PosixPath('/home/lbrenap/Documents/projects/legalpaca/corporate_reorganization/test_results/retrieval_ablation_local')

## SageMaker Processing run

Use the cell below if you want a managed Processing job.

By default this notebook runs only the dense systems remotely. To run `bm25_flat` remotely, switch to a custom or extended Processing image that already contains Java + Pyserini.

In [ ]:
processor = HuggingFaceProcessor(
    role=role,
    transformers_version="4.49.0",
    pytorch_version="2.5.1",
    py_version="py311",
    instance_type="ml.g5.12xlarge",
    instance_count=1,
)

processor.run(
    code="processing_eval/run_eval_sm.py",
    source_dir="../modernbert",
    inputs=[
        ProcessingInput(
            source=inputs["data"],
            destination="/opt/ml/processing/input/data",
        )
    ],
    outputs=[
        ProcessingOutput(
            source="/opt/ml/processing/output/eval",
            destination=processing_output_s3_uri,
        )
    ],
    arguments=[
        "--processed_dir", "/opt/ml/processing/input/data",
        "--output_dir", "/opt/ml/processing/output/eval",
        "--split", split,
        "--structured_model_s3_uri", structured_model_s3_uri,
        "--flat_model_s3_uri", flat_model_s3_uri,
        "--base_model_name_or_path", "answerdotai/ModernBERT-base",
        "--systems", processing_systems,
        "--regimes", ",".join(regimes),
        "--k_values", "1,5,10,20",
        "--max_len_query", "4096",
        "--max_len_passage", "600",
        "--query_batch_size", "64",
        "--passage_batch_size", "256",
    ],
    wait=True,
    logs=True,
)

print("Processing outputs:", processing_output_s3_uri)


## Outputs

Local run output directory:
- `results.json`
- `report.md`
- `config.json`
- `runs/rankings.jsonl`

Processing run output S3 directory contains the same files.